In [3]:
import pandas as pd
df = pd.read_csv('ibtracs.NA.list.v04r01.csv', skiprows=[1])
unique_storms = df['SID'].unique()
print(len(unique_storms))

2317


C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_10092\2761384259.py:2: DtypeWarning: Columns (3,19,20,23,24,172,173) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('ibtracs.NA.list.v04r01.csv', skiprows=[1])


In [4]:
import numpy as np
import pandas as pd

def _pick_col(cols, token, prefer=()):
    cols = [c.strip() for c in cols]
    for p in prefer:
        if p in cols:
            return p
    hits = [c for c in cols if token in c]
    return hits[0] if hits else None

def count_rankine_ready_storms(
    ibtracs_csv: str,
    wind_prefer=("USA_WIND", "WMO_WIND"),
    rmw_prefer=("USA_RMW", "WMO_RMW"),
    min_valid_rows_per_storm: int = 2,
):
    """
    Counts unique storms (SID) that have the minimum fields needed for:
      - crossing logic (time series with at least 2 valid fixes)
      - Rankine vortex winds (needs WIND and RMW at those fixes)

    Criteria for a storm to be counted:
      - at least `min_valid_rows_per_storm` rows where:
          SID, ISO_TIME, LAT, LON are present AND
          selected WIND column is finite AND selected RMW column is finite

    Returns: dict with counts + oldest storm info.
    """
    # IBTrACS "list" CSVs typically have a units row on line 2
    df = pd.read_csv(ibtracs_csv, skiprows=[1], low_memory=False)
    df.columns = df.columns.astype(str).str.strip()

    required_base = ["SID", "ISO_TIME", "LAT", "LON"]
    missing = [c for c in required_base if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Found: {df.columns[:40].tolist()}")

    wind_col = _pick_col(df.columns, "WIND", prefer=wind_prefer)
    rmw_col  = _pick_col(df.columns, "RMW",  prefer=rmw_prefer)

    if wind_col is None:
        raise ValueError("No WIND-like column found (expected e.g., USA_WIND or WMO_WIND).")
    if rmw_col is None:
        raise ValueError("No RMW-like column found (expected e.g., USA_RMW or WMO_RMW).")

    # Parse/clean
    df["SID"] = df["SID"].astype(str).str.strip()
    df["ISO_TIME"] = pd.to_datetime(df["ISO_TIME"], errors="coerce", utc=True)

    for c in ["LAT", "LON", wind_col, rmw_col]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[df[c].isin([-9999, -999, -99]), c] = np.nan

    # Keep only rows that can be used in crossing + Rankine
    ok = (
        df["SID"].notna()
        & df["ISO_TIME"].notna()
        & np.isfinite(df["LAT"])
        & np.isfinite(df["LON"])
        & np.isfinite(df[wind_col])
        & np.isfinite(df[rmw_col])
    )
    d0 = df.loc[ok, ["SID", "ISO_TIME", "LAT", "LON", wind_col, rmw_col] + (["NAME"] if "NAME" in df.columns else [])].copy()

    # Count valid rows per storm
    counts = d0.groupby("SID").size()
    good_sids = counts[counts >= min_valid_rows_per_storm].index

    n_total_storms = df["SID"].nunique()
    n_rankine_ready = len(good_sids)

    # Oldest storm among those
    d_good = d0[d0["SID"].isin(good_sids)].copy()
    first_time_by_sid = d_good.groupby("SID")["ISO_TIME"].min().sort_values()
    oldest_sid = first_time_by_sid.index[0]
    oldest_time = first_time_by_sid.iloc[0]

    oldest_name = None
    if "NAME" in df.columns:
        # grab a representative name from the earliest valid row for that SID
        row0 = df[(df["SID"] == oldest_sid)].copy()
        row0["ISO_TIME"] = pd.to_datetime(row0["ISO_TIME"], errors="coerce", utc=True)
        row0 = row0.sort_values("ISO_TIME")
        oldest_name = row0["NAME"].dropna().astype(str).str.strip().iloc[0] if row0["NAME"].notna().any() else None

    return {
        "wind_col": wind_col,
        "rmw_col": rmw_col,
        "min_valid_rows_per_storm": min_valid_rows_per_storm,
        "n_total_storms_in_file": int(n_total_storms),
        "n_storms_with_rankine_inputs": int(n_rankine_ready),
        "oldest_sid_with_rankine_inputs": oldest_sid,
        "oldest_time_utc_with_rankine_inputs": oldest_time,
        "oldest_name_with_rankine_inputs": oldest_name,
        "counts_valid_rows_per_storm": counts,  # pandas Series (optional to inspect)
    }

# ----------------------------
# Example usage
# ----------------------------
ib_csv = "ibtracs.NA.list.v04r01.csv"  # <-- change path if needed

out = count_rankine_ready_storms(
    ib_csv,
    wind_prefer=("USA_WIND", "WMO_WIND"),
    rmw_prefer=("USA_RMW", "WMO_RMW"),
    min_valid_rows_per_storm=2,   # needs at least 2 fixes for a crossing segment
)

print("Columns used:")
print("  wind_col:", out["wind_col"])
print("  rmw_col :", out["rmw_col"])
print("")
print("Counts:")
print("  total storms in file (unique SID):", out["n_total_storms_in_file"])
print("  storms with Rankine+crossing inputs:", out["n_storms_with_rankine_inputs"])
print(f"  criterion: >= {out['min_valid_rows_per_storm']} rows per storm with valid lat/lon/time/wind/rmw")
print("")
print("Oldest storm meeting criterion:")
print("  SID :", out["oldest_sid_with_rankine_inputs"])
print("  NAME:", out["oldest_name_with_rankine_inputs"])
print("  first valid time (UTC):", out["oldest_time_utc_with_rankine_inputs"])


Columns used:
  wind_col: USA_WIND
  rmw_col : USA_RMW

Counts:
  total storms in file (unique SID): 2317
  storms with Rankine+crossing inputs: 471
  criterion: >= 2 rows per storm with valid lat/lon/time/wind/rmw

Oldest storm meeting criterion:
  SID : 1858254N21273
  NAME: UNNAMED
  first valid time (UTC): 1858-09-16 17:00:00+00:00
